# Chapter 18 Lab: From Models to Systems (`ch12`)

This notebook explores the practical challenges of deploying ML models:
- Training-serving skew and the importance of reproducible preprocessing
- Thresholds as business policy decisions
- Monitoring for data drift and performance degradation
- Selective prediction and human-in-the-loop systems
- Feedback loops and their biasing effects

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Training-Serving Skew

Train a model with one scaler, then at serving time use a different scaler (fit on different data). Show how this causes accuracy to drop. Fix by saving and reusing the training scaler.

In [ ]:
np.random.seed(42)

# Generate training and test data
X_train, y_train = make_classification(
    n_samples=400,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=42
)

X_test, y_test = make_classification(
    n_samples=200,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=99
)

print("Training-Serving Skew Example")
print("=" * 60)

# CORRECT: Train with StandardScaler, save it, use same scaler at serving time
print(f"\n1. CORRECT: Using saved training scaler")
scaler_train = StandardScaler()
X_train_scaled_correct = scaler_train.fit_transform(X_train)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled_correct, y_train)

# At serving time: use the SAVED scaler
X_test_scaled_correct = scaler_train.transform(X_test)  # Use saved scaler
acc_correct = model.score(X_test_scaled_correct, y_test)
print(f"   Model accuracy: {acc_correct:.4f}")

# INCORRECT: Train with one scaler, but at serving time fit a NEW scaler on test data
print(f"\n2. INCORRECT: Fitting a new scaler at serving time")
scaler_serving = StandardScaler()
X_test_scaled_incorrect = scaler_serving.fit_transform(X_test)  # Fit on test data!
acc_incorrect = model.score(X_test_scaled_incorrect, y_test)
print(f"   Model accuracy: {acc_incorrect:.4f}")

print(f"\nAccuracy drop due to skew: {acc_correct - acc_incorrect:.4f}")
print(f"\nWhy this happens:")
print(f"  - Training scaler: mean={scaler_train.mean_[0]:.3f}, std={scaler_train.scale_[0]:.3f}")
print(f"  - Serving scaler:  mean={scaler_serving.mean_[0]:.3f}, std={scaler_serving.scale_[0]:.3f}")
print(f"  -> Different scalers transform test data differently!")
print(f"\nSolution: Always save the training scaler and reuse it at serving time.")

In [ ]:
# Visualize the skew
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

feature_idx = 0

# Original data distributions
ax = axes[0, 0]
ax.hist(X_train[:, feature_idx], bins=20, alpha=0.6, label='Train', color='steelblue', edgecolor='black')
ax.hist(X_test[:, feature_idx], bins=20, alpha=0.6, label='Test', color='orange', edgecolor='black')
ax.set_xlabel(f'Feature {feature_idx}')
ax.set_ylabel('Frequency')
ax.set_title('Original Data Distributions')
ax.legend()
ax.grid(alpha=0.3)

# Correctly scaled data
ax = axes[0, 1]
ax.hist(X_train_scaled_correct[:, feature_idx], bins=20, alpha=0.6, label='Train', color='steelblue', edgecolor='black')
ax.hist(X_test_scaled_correct[:, feature_idx], bins=20, alpha=0.6, label='Test (with saved scaler)', color='green', edgecolor='black')
ax.set_xlabel(f'Feature {feature_idx} (scaled)')
ax.set_ylabel('Frequency')
ax.set_title('CORRECT: Using Saved Training Scaler')
ax.legend()
ax.grid(alpha=0.3)

# Incorrectly scaled data
ax = axes[1, 0]
ax.hist(X_train_scaled_correct[:, feature_idx], bins=20, alpha=0.6, label='Train', color='steelblue', edgecolor='black')
ax.hist(X_test_scaled_incorrect[:, feature_idx], bins=20, alpha=0.6, label='Test (with new scaler)', color='red', edgecolor='black')
ax.set_xlabel(f'Feature {feature_idx} (scaled)')
ax.set_ylabel('Frequency')
ax.set_title('INCORRECT: Fitting New Scaler at Serving Time')
ax.legend()
ax.grid(alpha=0.3)

# Accuracy comparison
ax = axes[1, 1]
scenarios = ['Correct\n(Saved scaler)', 'Incorrect\n(New scaler)']
accuracies = [acc_correct, acc_incorrect]
colors = ['#2ca02c', '#d62728']
bars = ax.bar(scenarios, accuracies, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Impact of Training-Serving Skew', fontweight='bold')
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.02, f'{acc:.3f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Training-Serving Skew: The Cost of Inconsistent Preprocessing', 
             fontsize=13, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## 2. Threshold as Operating Policy

Train a model. Define two operating policies: "conservative" (high precision) and "aggressive" (high recall). Show that changing the threshold is a business decision, not a model change.

In [ ]:
np.random.seed(42)

# Generate data
X, y = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Get probability predictions
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Evaluate at different thresholds
thresholds = np.linspace(0.1, 0.9, 20)
precisions = []
recalls = []
accuracies = []

for thresh in thresholds:
    y_pred = (y_pred_proba >= thresh).astype(int)
    precisions.append(precision_score(y_test, y_pred, zero_division=0))
    recalls.append(recall_score(y_test, y_pred, zero_division=0))
    accuracies.append(accuracy_score(y_test, y_pred))

precisions = np.array(precisions)
recalls = np.array(recalls)
accuracies = np.array(accuracies)

# Two operating policies
conservative_thresh = 0.8  # High precision
aggressive_thresh = 0.3    # High recall

# Conservative
y_pred_conservative = (y_pred_proba >= conservative_thresh).astype(int)
prec_conservative = precision_score(y_test, y_pred_conservative, zero_division=0)
rec_conservative = recall_score(y_test, y_pred_conservative, zero_division=0)
acc_conservative = accuracy_score(y_test, y_pred_conservative)

# Aggressive
y_pred_aggressive = (y_pred_proba >= aggressive_thresh).astype(int)
prec_aggressive = precision_score(y_test, y_pred_aggressive, zero_division=0)
rec_aggressive = recall_score(y_test, y_pred_aggressive, zero_division=0)
acc_aggressive = accuracy_score(y_test, y_pred_aggressive)

print("Threshold as Operating Policy")
print("=" * 60)
print(f"\nConservative Policy (threshold = {conservative_thresh}):")
print(f"  Precision: {prec_conservative:.4f} (high)")
print(f"  Recall:    {rec_conservative:.4f} (low)")
print(f"  Accuracy:  {acc_conservative:.4f}")
print(f"  Use case: Minimize false positives (e.g., fraud detection)")

print(f"\nAggressive Policy (threshold = {aggressive_thresh}):")
print(f"  Precision: {prec_aggressive:.4f} (low)")
print(f"  Recall:    {rec_aggressive:.4f} (high)")
print(f"  Accuracy:  {acc_aggressive:.4f}")
print(f"  Use case: Minimize false negatives (e.g., disease screening)")

print(f"\nKey insight:")
print(f"  The model doesn't change. Only the threshold changes.")
print(f"  The threshold is a BUSINESS DECISION, not a model decision.")

In [ ]:
# Visualize precision-recall tradeoff
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision-recall curve
ax = axes[0]
ax.plot(recalls, precisions, 'o-', linewidth=2, markersize=5, color='steelblue')
ax.plot(rec_conservative, prec_conservative, 'D', markersize=12, color='#2ca02c', label='Conservative (threshold=0.8)')
ax.plot(rec_aggressive, prec_aggressive, 's', markersize=12, color='#d62728', label='Aggressive (threshold=0.3)')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Tradeoff', fontweight='bold')
ax.set_xlim([0, 1.0])
ax.set_ylim([0, 1.0])
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Metrics vs threshold
ax = axes[1]
ax.plot(thresholds, precisions, 'o-', linewidth=2, markersize=5, label='Precision', color='steelblue')
ax.plot(thresholds, recalls, 's-', linewidth=2, markersize=5, label='Recall', color='orange')
ax.plot(thresholds, accuracies, '^-', linewidth=2, markersize=5, label='Accuracy', color='green')
ax.axvline(conservative_thresh, color='#2ca02c', linestyle='--', linewidth=2, alpha=0.5, label='Conservative')
ax.axvline(aggressive_thresh, color='#d62728', linestyle='--', linewidth=2, alpha=0.5, label='Aggressive')
ax.set_xlabel('Decision Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Metrics vs. Threshold', fontweight='bold')
ax.set_xlim([0.1, 0.9])
ax.set_ylim([0, 1.0])
ax.legend(fontsize=9, loc='best')
ax.grid(alpha=0.3)

plt.suptitle('Operating Policy: How Threshold Changes Model Behavior', 
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 3. Monitoring: Detecting Degradation

Simulate a deployed model receiving batches of data over 20 weeks. Data drifts starting at week 10. Plot rolling accuracy and alert when it falls below threshold.

In [ ]:
np.random.seed(42)

# Train a model on baseline data
X_baseline, y_baseline = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=42
)

scaler = StandardScaler()
X_baseline_scaled = scaler.fit_transform(X_baseline)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_baseline_scaled, y_baseline)

# Simulate 20 weeks of data
n_weeks = 20
batch_size = 100
weekly_accuracies = []
week_numbers = []

for week in range(1, n_weeks + 1):
    # Before week 10: same distribution as baseline
    if week < 10:
        X_week, y_week = make_classification(
            n_samples=batch_size,
            n_features=10,
            n_informative=8,
            n_redundant=2,
            random_state=42 + week
        )
    else:
        # After week 10: distribution shift (drift)
        X_week, y_week = make_classification(
            n_samples=batch_size,
            n_features=10,
            n_informative=8,
            n_redundant=2,
            random_state=42 + week
        )
        # Add drift: shift mean and increase noise
        X_week = X_week + 0.5 * (week - 10) / 10  # Progressive shift
        X_week = X_week + 0.1 * (week - 10) / 10 * np.random.randn(*X_week.shape)  # Additional noise
    
    X_week_scaled = scaler.transform(X_week)
    acc = model.score(X_week_scaled, y_week)
    weekly_accuracies.append(acc)
    week_numbers.append(week)

weekly_accuracies = np.array(weekly_accuracies)

# Compute rolling accuracy (3-week window)
rolling_window = 3
rolling_accuracies = np.convolve(weekly_accuracies, np.ones(rolling_window) / rolling_window, mode='valid')
rolling_weeks = week_numbers[rolling_window - 1:]

# Alert threshold
alert_threshold = 0.80
alerts = []
for week, rolling_acc in zip(rolling_weeks, rolling_accuracies):
    if rolling_acc < alert_threshold:
        alerts.append((week, rolling_acc))

print("Monitoring: Detecting Degradation")
print("=" * 60)
print(f"\nBaseline accuracy: {weekly_accuracies[:9].mean():.4f}")
print(f"Post-drift accuracy: {weekly_accuracies[9:].mean():.4f}")
print(f"Accuracy drop: {weekly_accuracies[:9].mean() - weekly_accuracies[9:].mean():.4f}")

print(f"\nAlert threshold: {alert_threshold:.4f}")
if alerts:
    print(f"Alerts triggered at weeks:")
    for week, rolling_acc in alerts:
        print(f"  Week {week:2d}: rolling accuracy = {rolling_acc:.4f}")
else:
    print(f"No alerts triggered.")

first_alert = alerts[0][0] if alerts else None
if first_alert:
    print(f"\nFirst alert at week {first_alert} (data drift started at week 10)")
    print(f"Detection lag: {first_alert - 10} weeks")

In [ ]:
# Visualize monitoring
fig, ax = plt.subplots(figsize=(12, 6))

# Weekly accuracy
ax.plot(week_numbers, weekly_accuracies, 'o-', linewidth=2, markersize=6, label='Weekly accuracy', color='steelblue')

# Rolling accuracy
ax.plot(rolling_weeks, rolling_accuracies, 's-', linewidth=2.5, markersize=6, label='Rolling accuracy (3-week)', color='orange')

# Alert threshold
ax.axhline(alert_threshold, color='red', linestyle='--', linewidth=2, label=f'Alert threshold ({alert_threshold:.2f})', alpha=0.7)

# Drift period
ax.axvspan(9.5, 20, alpha=0.1, color='red', label='Data drift period')
ax.text(14.5, 0.95, 'Data Drift\n(Week 10+)', ha='center', fontsize=11, fontweight='bold', 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Mark alerts
if alerts:
    for week, rolling_acc in alerts:
        ax.plot(week, rolling_acc, 'X', markersize=15, color='darkred', markeredgewidth=2, markeredgecolor='black')
    ax.text(alerts[0][0], alerts[0][1] - 0.05, 'ALERT', ha='center', fontsize=10, fontweight='bold', color='darkred')

ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Monitoring: Detecting Performance Degradation Due to Data Drift', fontweight='bold', fontsize=13)
ax.set_xlim([0, 21])
ax.set_ylim([0.65, 1.0])
ax.legend(fontsize=11, loc='lower left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Selective Prediction (Human-in-the-Loop)

Model predicts confidently but defers to a human when confidence < 0.7. Show that deferred cases are the hardest. Plot accuracy on auto-predicted vs. deferred vs. coverage tradeoff.

In [ ]:
np.random.seed(42)

# Generate data
X, y = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Get predictions and confidence
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
y_pred = model.predict(X_test_scaled)
confidence = np.maximum(y_pred_proba, 1 - y_pred_proba)

# Selective prediction thresholds
confidence_thresholds = np.linspace(0.5, 0.95, 20)
coverage_vals = []
accuracy_auto_vals = []
accuracy_deferred_vals = []

for thresh in confidence_thresholds:
    auto_mask = confidence >= thresh
    deferred_mask = ~auto_mask
    
    coverage = auto_mask.sum() / len(y_test)
    coverage_vals.append(coverage)
    
    if auto_mask.sum() > 0:
        acc_auto = (y_pred[auto_mask] == y_test[auto_mask]).mean()
        accuracy_auto_vals.append(acc_auto)
    else:
        accuracy_auto_vals.append(np.nan)
    
    if deferred_mask.sum() > 0:
        # Assume human gets deferred cases right
        acc_deferred = 1.0  # Human accuracy
        accuracy_deferred_vals.append(acc_deferred)
    else:
        accuracy_deferred_vals.append(np.nan)

coverage_vals = np.array(coverage_vals)
accuracy_auto_vals = np.array(accuracy_auto_vals)
accuracy_deferred_vals = np.array(accuracy_deferred_vals)

# Specific threshold: 0.7
confidence_threshold = 0.7
auto_mask = confidence >= confidence_threshold
deferred_mask = ~auto_mask

coverage = auto_mask.sum() / len(y_test)
acc_auto = (y_pred[auto_mask] == y_test[auto_mask]).mean()
mean_confidence_auto = confidence[auto_mask].mean()
mean_confidence_deferred = confidence[deferred_mask].mean()

# Overall accuracy with selective prediction
overall_acc_selective = (auto_mask.sum() * acc_auto + deferred_mask.sum() * 1.0) / len(y_test)
overall_acc_auto = (y_pred == y_test).mean()

print("Selective Prediction (Human-in-the-Loop)")
print("=" * 60)
print(f"\nModel accuracy (auto on all): {overall_acc_auto:.4f}")

print(f"\nWith selective prediction (confidence threshold = {confidence_threshold}):")
print(f"  Auto-predicted cases: {auto_mask.sum()} ({coverage:.1%})")
print(f"    Mean confidence: {mean_confidence_auto:.4f}")
print(f"    Accuracy: {acc_auto:.4f}")

print(f"\n  Deferred cases: {deferred_mask.sum()} ({1-coverage:.1%})")
print(f"    Mean confidence: {mean_confidence_deferred:.4f}")
print(f"    Accuracy (human): 1.0000 (assumed)")

print(f"\n  Overall accuracy: {overall_acc_selective:.4f}")
print(f"  Improvement over auto-only: {overall_acc_selective - overall_acc_auto:+.4f}")

print(f"\nKey insight:")
print(f"  Deferred cases have LOW confidence (harder cases).")
print(f"  By deferring to humans, we improve overall accuracy.")
print(f"  This is the accuracy-coverage tradeoff.")

In [ ]:
# Visualize selective prediction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution (auto vs deferred)
ax = axes[0]
ax.hist(confidence[auto_mask], bins=15, alpha=0.7, label='Auto-predicted', color='green', edgecolor='black')
ax.hist(confidence[deferred_mask], bins=15, alpha=0.7, label='Deferred (to human)', color='red', edgecolor='black')
ax.axvline(confidence_threshold, color='blue', linestyle='--', linewidth=2, label=f'Threshold ({confidence_threshold})')
ax.set_xlabel('Model Confidence', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Confidence Distribution: Auto vs. Deferred', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Accuracy vs coverage
ax = axes[1]
ax.plot(coverage_vals, accuracy_auto_vals, 'o-', linewidth=2, markersize=6, label='Auto accuracy', color='orange')
ax.axhline(overall_acc_auto, color='blue', linestyle=':', linewidth=2, alpha=0.5, label='Auto-only overall')

# Compute overall accuracy with human on deferred
overall_accs_selective = (coverage_vals * accuracy_auto_vals + (1 - coverage_vals) * 1.0)
ax.plot(coverage_vals, overall_accs_selective, 's-', linewidth=2.5, markersize=6, label='Overall (with human)', color='green')

# Mark the selected threshold
idx_selected = np.argmin(np.abs(coverage_vals - coverage))
ax.plot(coverage, accuracy_auto_vals[idx_selected], 'D', markersize=12, color='darkgreen', markeredgewidth=2, markeredgecolor='black', label=f'Selected (threshold={confidence_threshold})')

ax.set_xlabel('Coverage (fraction auto-predicted)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy-Coverage Tradeoff', fontweight='bold')
ax.set_xlim([0.4, 1.0])
ax.set_ylim([0.75, 1.01])
ax.legend(fontsize=10, loc='lower left')
ax.grid(alpha=0.3)

plt.suptitle('Selective Prediction: Human-in-the-Loop Systems', 
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 5. Feedback Loops and Bias Amplification

Model recommends items. Users click recommended items more (popularity bias). Retrain on click data. Show the model becomes more biased over 5 rounds.

In [ ]:
np.random.seed(42)

# Simulate a recommendation system
n_items = 50
n_users = 1000
n_rounds = 5

# True item quality: random
true_quality = np.random.rand(n_items)

# Track metrics over rounds
avg_quality_recommended = []
diversity_recommended = []  # Entropy of recommendations

# Initial: recommend items uniformly at random
recommendation_probs = np.ones(n_items) / n_items

for round_num in range(n_rounds):
    # Recommend items based on current probabilities
    n_recommendations = 200
    recommendations = np.random.choice(n_items, n_recommendations, p=recommendation_probs)
    
    # Users click based on recommendation frequency (popularity bias)
    # More recommended items = more clicks
    # True signal: item quality
    click_probs = recommendation_probs * 0.7 + true_quality / true_quality.sum() * 0.3
    
    clicks = np.random.choice(n_items, 100, p=click_probs / click_probs.sum())
    
    # Retrain: recommend items based on click frequency
    click_counts = np.bincount(clicks, minlength=n_items)
    recommendation_probs = click_counts / click_counts.sum() + 1e-6  # Add small constant to avoid zeros
    recommendation_probs = recommendation_probs / recommendation_probs.sum()
    
    # Compute metrics
    recommended_items = recommendations
    avg_quality = true_quality[recommended_items].mean()
    avg_quality_recommended.append(avg_quality)
    
    # Diversity: entropy of recommendations
    diversity = -np.sum(recommendation_probs[recommendation_probs > 0] * np.log(recommendation_probs[recommendation_probs > 0]))
    diversity_recommended.append(diversity)

avg_quality_recommended = np.array(avg_quality_recommended)
diversity_recommended = np.array(diversity_recommended)

print("Feedback Loops and Bias Amplification")
print("=" * 60)

print(f"\nAverage quality of recommended items:")
for round_num, quality in enumerate(avg_quality_recommended, 1):
    print(f"  Round {round_num}: {quality:.4f}")

print(f"\nDiversity (entropy) of recommendations:")
for round_num, diversity in enumerate(diversity_recommended, 1):
    print(f"  Round {round_num}: {diversity:.4f}")

print(f"\nObservation:")
print(f"  Quality stays roughly the same (true signal is weak).")
print(f"  Diversity DECREASES over rounds (bias amplification).")
print(f"  Model focuses on popular items, ignoring long tail.")

print(f"\nWhy this happens:")
print(f"  1. Initial model recommends some popular items")
print(f"  2. Users click on recommended items more (popularity bias)")
print(f"  3. We retrain on clicks, which reinforces popular items")
print(f"  4. Cycle repeats: popularity amplifies")

print(f"\nSolution:")
print(f"  - Separate feedback loop (implicit feedback) from data collection")
print(f"  - Use exploration (recommend some random items to gather true signal)")
print(f"  - Regularize toward true quality (if available)")

In [ ]:
# Visualize feedback loops
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Quality over rounds
ax = axes[0]
rounds = np.arange(1, n_rounds + 1)
ax.plot(rounds, avg_quality_recommended, 'o-', linewidth=2.5, markersize=8, color='steelblue', label='Avg quality of recommendations')
ax.axhline(true_quality.mean(), color='green', linestyle='--', linewidth=2, label='True average quality')
ax.set_xlabel('Training Round', fontsize=11)
ax.set_ylabel('Average Quality', fontsize=11)
ax.set_title('Quality of Recommendations Over Time', fontweight='bold')
ax.set_xticks(rounds)
ax.set_ylim([0.4, 0.65])
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Diversity over rounds
ax = axes[1]
ax.plot(rounds, diversity_recommended, 's-', linewidth=2.5, markersize=8, color='#d62728')
ax.fill_between(rounds, diversity_recommended, alpha=0.3, color='#d62728')
ax.set_xlabel('Training Round', fontsize=11)
ax.set_ylabel('Diversity (Entropy)', fontsize=11)
ax.set_title('Diversity of Recommendations Over Time', fontweight='bold')
ax.set_xticks(rounds)
ax.grid(alpha=0.3)

# Add annotation
ax.annotate('Diversity\ncollapses', xy=(n_rounds, diversity_recommended[-1]),
            xytext=(n_rounds - 1.5, diversity_recommended[-1] + 0.3),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=11, fontweight='bold', color='red')

plt.suptitle('Feedback Loops Amplify Popularity Bias', 
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## What to Notice

1. **Training-Serving Skew Is Serious**: Using a different scaler at serving time (or any preprocessing mismatch) causes accuracy to drop. Always save and reuse training artifacts.

2. **Thresholds Are Business Decisions**: The model is fixed; only the threshold changes. Precision-recall tradeoff is built into every classifier. Choose the threshold based on business costs, not model metrics.

3. **Monitoring Is Essential**: Performance degrades silently. Set up automated monitoring for accuracy, data drift, and calibration. Alert when metrics fall below acceptable thresholds.

4. **Selective Prediction Works**: Humans and models have complementary strengths. Defer to humans on low-confidence predictions. This improves overall accuracy while maintaining coverage.

5. **Feedback Loops Amplify Bias**: When you retrain on model predictions, you amplify biases. Initial popularity → more recommendations → more clicks → amplified popularity. Break the loop with exploration or external signal.

6. **Systems Thinking Matters**: ML deployment is not just about model accuracy. It's about preprocessing, monitoring, human-in-the-loop workflows, and feedback loop management.